# Iceberg vs Hive on the same Parquet files: run four schema evolutions and watch what survives

Your team has been writing Hive-style external tables for years. The platform lead wants everyone to migrate to Iceberg. You are skeptical. The product manager told the room last week that the schema is changing every six weeks for the next two quarters. Today you settle the format question with data.

Same Parquet files. Two catalog entries. Four schema evolutions you will actually face on the job: add a column, rename a column, drop a column, reorder columns. Run each evolution on both formats. Record what survived. The lab does not pick a winner. You do, in the reflection cell, given the constraint scenario the PM just sent.

Four checkpoints map to those four evolutions:

1. Both `events_hive` and `events_iceberg` exist over the same Parquet seed and return identical row counts.
2. Add the `loyalty_tier` column on both sides. Backfill behavior recorded.
3. Rename `order_total` to `order_amount_usd` on both sides. Hive renames at the metastore but reads break, because Parquet stores by name. Iceberg's metadata layer makes the rename transparent.
4. Drop `legacy_session_id` and reorder columns on both sides. Aggregate every result into `outcome/results.json` and print the comparison metric.

**Time.** About 60 minutes hands-on. Session window 90 minutes. Athena DDL returns in seconds; the only wall-clock cost is the CTAS that seeds the Parquet at the start.

**Cost.** This lab costs about 20 to 50 cents per session. Athena scans are tiny (under 100 MB total across all queries), Glue catalog operations are free, S3 storage is rounding-error. The CTAS that builds the Parquet seed is the only material cost. Your morning coffee cost more than this lab will.

This lab maps to the Data Engineering job-prep track, sub-track B: Warehouse and Lakehouse Mastery.

In [ ]:
# NBVAL_SKIP
# Install labrun-checks. Pinned to a specific version per LAB-CREATION-BLUEPRINT
# build rules. Never use unpinned installs.

!pip install --quiet labrun-checks==0.8.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern from LAB-CREATION-BLUEPRINT
# section 12. Region is us-east-1 per Section 15.

import atexit
import getpass
import json
import time

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupResource,
)

LAB_ID = "iceberg-vs-hive-format"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID  # must equal the cert YAML labs[].slug exactly
REGION = "us-east-1"  # all track-data-engineering labs run in us-east-1

# Resource identifiers. The bucket name is finalized after STS confirms the
# account ID (S3 bucket names must be globally unique).
BUCKET_NAME = None
DATABASE_NAME = f"labrun_{LAB_ID.replace('-', '_')}_db"  # Glue/Athena reject hyphens in identifiers
WORKGROUP_NAME = f"labrun-{LAB_ID}-wg"
TABLE_HIVE = "events_hive"
TABLE_ICEBERG = "events_iceberg"

# S3 prefixes. parquet/ holds the seed Parquet files that both tables point
# at; iceberg/ holds Iceberg's data + metadata; outcome/ holds the comparison
# results JSON; athena-results/ is the workgroup output sink.
PARQUET_PREFIX = "parquet/"
ICEBERG_PREFIX = "iceberg/"
OUTCOME_PREFIX = "outcome/"
OUTCOME_KEY = f"{OUTCOME_PREFIX}results.json"
ATHENA_RESULTS_PREFIX = "athena-results/"

In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials via STS GetCallerIdentity,
# and confirm the region. This cell must succeed before the manifest cell
# creates anything per LAB-CREATION-BLUEPRINT section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()
region_input = input(f"AWS region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported for this lab.")
    print(f"All track-data-engineering labs run in {REGION} (N. Virginia).")
    print("Re-run this cell and accept the default.")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

# Validate credentials against AWS via STS GetCallerIdentity. Fail fast with a
# clear message rather than waiting for the first DDL call to error out.
sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    print()
    print("Refresh your AWS credentials and re-run this cell.")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")
print(f"Session token in use: {bool(aws_session_token)}")

# Resolve the bucket name now that the account ID is known.
BUCKET_NAME = f"labrun-{LAB_ID}-{ACCOUNT_ID}"
print(f"Bucket name resolved: {BUCKET_NAME}")

# Register the session with labrun-checks. register() returns None; do not
# assign its return value.
register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan.
# Reverse-creation order. No critical (hourly-billed) resources in this lab.
# The iceberg_table entry's S3 location is encoded inside cli_delete_command
# because CleanupResource does not expose a location field; the adapter
# infers the metadata + data prefix from the table's StorageDescriptor at
# delete time. Per RESOURCE-SAFETY-SPEC Rule 4, orphan scan blocks execution
# if any tagged resources from a prior session exist.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="athena_workgroup",
        id=WORKGROUP_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws athena delete-work-group --work-group {WORKGROUP_NAME} "
            f"--recursive-delete-option"
        ),
    ),
    # Iceberg table location: s3://{BUCKET_NAME}/{ICEBERG_PREFIX}
    # The iceberg_table adapter drops the Glue catalog entry AND removes
    # data + metadata under the iceberg/ prefix.
    CleanupResource(
        type="iceberg_table",
        id=f"{DATABASE_NAME}.{TABLE_ICEBERG}",
        region=REGION,
        cli_delete_command=(
            f"aws glue delete-table --database-name {DATABASE_NAME} --name {TABLE_ICEBERG} && "
            f"aws s3 rm s3://{BUCKET_NAME}/{ICEBERG_PREFIX} --recursive"
        ),
    ),
    CleanupResource(
        type="glue_table",
        id=f"{DATABASE_NAME}.{TABLE_HIVE}",
        region=REGION,
        cli_delete_command=f"aws glue delete-table --database-name {DATABASE_NAME} --name {TABLE_HIVE}",
    ),
    CleanupResource(
        type="glue_database",
        id=DATABASE_NAME,
        region=REGION,
        cli_delete_command=f"aws glue delete-database --name {DATABASE_NAME}",
    ),
    CleanupResource(
        type="s3_bucket",
        id=BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
    ),
]


def _atexit_cleanup() -> None:
    """Best-effort teardown on kernel shutdown.

    Runs run_cleanup against the manifest. Errors are swallowed because
    atexit handlers must not raise; the dedicated cleanup cell prints the
    full structured report and is the authoritative cleanup entry point.
    """
    try:
        run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list[str]:
    """Refuse to start if a previous run left tagged resources behind.

    Per RESOURCE-SAFETY-SPEC Rule 4, the setup cell must check for leftover
    state with the lab's tag and surface the cleanup command before creating
    any new resources.
    """
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns: list[str] = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("These are leftovers from a previous run of this lab. Re-running")
    print("setup against an unclean state can produce duplicate resources.")
    print("Run the cleanup cell at the bottom of this notebook first, or")
    print("remove them manually with the AWS CLI commands printed by the")
    print("cleanup cell, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

In [ ]:
# NBVAL_SKIP
# Athena query helper. Submits a query against the lab workgroup, polls
# until SUCCEEDED / FAILED / CANCELLED, and returns the full QueryExecution
# dict. Quirky wait messages keep the polling loop honest about what is
# happening underneath. Used by every task that runs SQL.

athena_client = boto3.client(
    "athena",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

_WAIT_MESSAGES = [
    "Athena is grading your DDL. Hold tight...",
    "Iceberg is rewriting its metadata snapshot. About five seconds...",
    "Hive is renaming at the metastore but the Parquet column names did not get the memo...",
    "Asking Athena nicely for your row count...",
    "Glue catalog is opening the right drawer...",
]


def run_athena(sql: str, deadline_seconds: int = 180, _msg_index: list = [0]) -> dict:
    """Run a SQL statement on the lab workgroup. Block until terminal state.

    Returns the QueryExecution dict on success; raises RuntimeError on a
    non-SUCCEEDED terminal state so callers see the StateChangeReason.
    """
    start = athena_client.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={"Database": DATABASE_NAME},
        WorkGroup=WORKGROUP_NAME,
    )
    qid = start["QueryExecutionId"]

    msg = _WAIT_MESSAGES[_msg_index[0] % len(_WAIT_MESSAGES)]
    _msg_index[0] += 1
    print(f"  {msg}")

    deadline = time.time() + deadline_seconds
    while time.time() < deadline:
        resp = athena_client.get_query_execution(QueryExecutionId=qid)
        state = resp["QueryExecution"]["Status"]["State"]
        if state in ("SUCCEEDED", "FAILED", "CANCELLED"):
            if state != "SUCCEEDED":
                reason = resp["QueryExecution"]["Status"].get("StateChangeReason", "(no reason)")
                raise RuntimeError(f"Athena query {qid} ended in {state}: {reason}\nSQL: {sql}")
            return resp["QueryExecution"]
        time.sleep(2)
    raise RuntimeError(f"Athena query {qid} did not finish within {deadline_seconds}s.")


def fetch_rows(query_execution_id: str) -> list[list[str]]:
    """Return the result rows of a finished query as a list of cell-string rows.

    Includes the header row at index 0. Adequate for the small results this
    lab inspects (under 100 rows in every checkpoint).
    """
    paginator = athena_client.get_paginator("get_query_results")
    rows: list[list[str]] = []
    for page in paginator.paginate(QueryExecutionId=query_execution_id):
        for row in page["ResultSet"]["Rows"]:
            cells = [c.get("VarCharValue", "") for c in row["Data"]]
            rows.append(cells)
    return rows


print("Athena helper ready. run_athena() and fetch_rows() loaded.")

## Task 1: Stand up both formats over the same Parquet files

The lab needs two real catalog entries that disagree about who owns the data layout. Athena and Glue power both.

Build it in this order:

1. Create the S3 bucket, tag it with `labrun:lab-slug=iceberg-vs-hive-format`, and put the Athena workgroup in place. The workgroup MUST run Athena Engine version 3 (`EngineVersion='AthenaEngineVersion3'`), because Iceberg DDL is only supported on v3. A workgroup defaulted to v2 is the most common Task 1 failure mode.
2. Create the Glue database (`labrun_iceberg_vs_hive_format_db`). Glue catalog identifiers cannot contain hyphens, so the lab uses underscores.
3. Seed a small Parquet dataset (about 1,000 rows) at `s3://{bucket}/parquet/` via Athena CTAS. The seed has six columns: `event_id` (bigint), `customer_id` (bigint), `event_ts` (timestamp), `country` (string), `order_total` (double), `legacy_session_id` (string). This is the canonical events shape the team has been writing for years.
4. Create `events_hive` as a `CREATE EXTERNAL TABLE` over the `parquet/` prefix. This is the Hive-style catalog entry your team has been using.
5. Create `events_iceberg` as a `CREATE TABLE ... USING ICEBERG` table at `s3://{bucket}/iceberg/`, populated via `INSERT INTO ... SELECT FROM events_hive`. This is the Iceberg catalog entry over the same logical data. Iceberg manages its own file layout under the `iceberg/` prefix; that is what makes the format comparison meaningful.

After this task, both tables have the same row count and the same six columns. Everything that follows is metadata divergence on top of identical data.

In [ ]:
# NBVAL_SKIP
# Task 1: Bucket + tag + Athena workgroup (Engine v3) + Glue database + seed
# Parquet via CTAS + events_hive + events_iceberg.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
glue = boto3.client(
    "glue",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# 1. Create the bucket. us-east-1 rejects LocationConstraint; the lab pins
# region so the simple form is correct.
try:
    # YOUR CODE: Create the S3 bucket using s3.create_bucket(Bucket=BUCKET_NAME).
    print(f"Bucket created: {BUCKET_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] in ("BucketAlreadyOwnedByYou", "BucketAlreadyExists"):
        print(f"Bucket {BUCKET_NAME} already owned; reusing.")
    else:
        raise

s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)
print(f"Bucket tagged: {LAB_TAG_KEY}={LAB_TAG_VALUE}")

# 2. Athena workgroup pinned to Engine v3 (Iceberg DDL requires it).
athena_results_uri = f"s3://{BUCKET_NAME}/{ATHENA_RESULTS_PREFIX}"
try:
    # YOUR CODE: Create the workgroup using athena_client.create_work_group().
    # Required Configuration keys:
    #   - ResultConfiguration: {"OutputLocation": athena_results_uri}
    #   - EnforceWorkGroupConfiguration: True
    #   - EngineVersion: {"SelectedEngineVersion": "Athena engine version 3"}
    #   - PublishCloudWatchMetricsEnabled: False
    # Pass Name=WORKGROUP_NAME, Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}].
    print(f"Athena workgroup created: {WORKGROUP_NAME} (Engine v3)")
except ClientError as e:
    if e.response["Error"]["Code"] == "InvalidRequestException":
        athena_client.update_work_group(
            WorkGroup=WORKGROUP_NAME,
            ConfigurationUpdates={
                "ResultConfigurationUpdates": {"OutputLocation": athena_results_uri},
                "EnforceWorkGroupConfiguration": True,
                "EngineVersion": {"SelectedEngineVersion": "Athena engine version 3"},
            },
            State="ENABLED",
        )
        print(f"Athena workgroup {WORKGROUP_NAME} already existed; updated to Engine v3.")
    else:
        raise

# 3. Glue database. Free, control-plane.
try:
    glue.create_database(
        DatabaseInput={
            "Name": DATABASE_NAME,
            "Description": f"Catalog for {LAB_ID}",
        },
    )
    glue.tag_resource(
        ResourceArn=f"arn:aws:glue:{REGION}:{ACCOUNT_ID}:database/{DATABASE_NAME}",
        TagsToAdd={LAB_TAG_KEY: LAB_TAG_VALUE},
    )
    print(f"Glue database created and tagged: {DATABASE_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] == "AlreadyExistsException":
        print(f"Glue database {DATABASE_NAME} already exists; reusing.")
    else:
        raise

# 4. Seed Parquet via CTAS into a transient seed table; then drop the seed
# table so the Parquet files at parquet/ are the only artifact left. CTAS
# is the cheapest path to a real Parquet dataset on this lab's scale.
SEED_TABLE = "events_seed_ctas"

# Drop any prior seed table so the CTAS can re-create cleanly across re-runs.
try:
    glue.delete_table(DatabaseName=DATABASE_NAME, Name=SEED_TABLE)
except ClientError as e:
    if e.response["Error"]["Code"] != "EntityNotFoundException":
        raise
# Clear the parquet prefix from any prior partial run so CTAS does not fail
# on "location not empty".
paginator = s3.get_paginator("list_objects_v2")
to_delete = []
for page in paginator.paginate(Bucket=BUCKET_NAME, Prefix=PARQUET_PREFIX):
    for obj in page.get("Contents", []):
        to_delete.append({"Key": obj["Key"]})
if to_delete:
    s3.delete_objects(Bucket=BUCKET_NAME, Delete={"Objects": to_delete})

seed_ctas_sql = f"""
CREATE TABLE {DATABASE_NAME}.{SEED_TABLE}
WITH (
  format = 'PARQUET',
  external_location = 's3://{BUCKET_NAME}/{PARQUET_PREFIX}'
) AS
SELECT
  CAST(seq AS bigint) AS event_id,
  CAST(1000 + (seq % 250) AS bigint) AS customer_id,
  from_iso8601_timestamp(
    concat('2026-04-', lpad(CAST(1 + (seq % 28) AS varchar), 2, '0'),
           'T', lpad(CAST(seq % 24 AS varchar), 2, '0'), ':00:00Z')
  ) AS event_ts,
  CASE seq % 5
    WHEN 0 THEN 'US' WHEN 1 THEN 'CA' WHEN 2 THEN 'UK'
    WHEN 3 THEN 'DE' ELSE 'JP' END AS country,
  CAST(10.0 + (seq % 200) * 1.37 AS double) AS order_total,
  concat('sess_', lpad(CAST(seq AS varchar), 6, '0')) AS legacy_session_id
FROM (
  SELECT sequence(1, 1000) AS s
) t
CROSS JOIN UNNEST(s) AS u(seq)
"""

# YOUR CODE: Run the seed CTAS by calling run_athena(seed_ctas_sql).
# This is the only material-cost step in the lab; it scans roughly nothing
# (the rows come from sequence()) but it materializes ~1000 Parquet rows.
print(f"Seeded ~1000 Parquet rows at s3://{BUCKET_NAME}/{PARQUET_PREFIX}")

# Drop the seed-CTAS catalog entry; the Parquet files at parquet/ stay.
glue.delete_table(DatabaseName=DATABASE_NAME, Name=SEED_TABLE)

# 5. events_hive over the same Parquet prefix as a CREATE EXTERNAL TABLE.
hive_ddl = f"""
CREATE EXTERNAL TABLE {DATABASE_NAME}.{TABLE_HIVE} (
  event_id bigint,
  customer_id bigint,
  event_ts timestamp,
  country string,
  order_total double,
  legacy_session_id string
)
STORED AS PARQUET
LOCATION 's3://{BUCKET_NAME}/{PARQUET_PREFIX}'
"""
# YOUR CODE: Run hive_ddl using run_athena().
print(f"Hive external table created: {DATABASE_NAME}.{TABLE_HIVE}")

# 6. events_iceberg over s3://{BUCKET}/iceberg/. Create it empty with the
# Iceberg DDL, then INSERT the same rows from events_hive. Athena Engine v3
# supports CREATE TABLE ... USING ICEBERG via table_type='ICEBERG'.
iceberg_ddl = f"""
CREATE TABLE {DATABASE_NAME}.{TABLE_ICEBERG} (
  event_id bigint,
  customer_id bigint,
  event_ts timestamp,
  country string,
  order_total double,
  legacy_session_id string
)
LOCATION 's3://{BUCKET_NAME}/{ICEBERG_PREFIX}'
TBLPROPERTIES ('table_type' = 'ICEBERG', 'format' = 'PARQUET')
"""
# YOUR CODE: Run iceberg_ddl using run_athena().
print(f"Iceberg table created: {DATABASE_NAME}.{TABLE_ICEBERG}")

# Populate Iceberg from the Hive table so both tables have identical rows.
insert_sql = (
    f"INSERT INTO {DATABASE_NAME}.{TABLE_ICEBERG} "
    f"SELECT * FROM {DATABASE_NAME}.{TABLE_HIVE}"
)
# YOUR CODE: Run insert_sql using run_athena().
print(f"Iceberg table populated from {TABLE_HIVE}")

print()
print("Both tables stand over the same logical data. Schema evolutions start in Task 2.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: Both tables exist over the same logical data and return
# matching row counts. No comparison logic; this is straight infrastructure
# validation per Compare-it pattern (blueprint Section 21).

def checkpoint_1(session):
    try:
        glue_client = boto3.client(
            "glue",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        # Workgroup is on Engine v3.
        try:
            wg = athena_client.get_work_group(WorkGroup=WORKGROUP_NAME)
        except ClientError as e:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Athena workgroup {WORKGROUP_NAME!r} not found: "
                    f"{e.response['Error']['Code']}. Iceberg DDL requires "
                    f"this workgroup to exist and run Athena Engine v3."
                ),
            )
        engine = wg["WorkGroup"]["Configuration"].get("EngineVersion", {})
        effective = engine.get("EffectiveEngineVersion", "")
        if "3" not in effective:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Athena workgroup {WORKGROUP_NAME!r} is on engine "
                    f"{effective!r}. Iceberg DDL requires Athena Engine v3. "
                    f"Update the workgroup's SelectedEngineVersion to "
                    f"'Athena engine version 3'."
                ),
            )

        # Both tables exist in the Glue catalog.
        for tbl in (TABLE_HIVE, TABLE_ICEBERG):
            try:
                glue_client.get_table(DatabaseName=DATABASE_NAME, Name=tbl)
            except ClientError as e:
                if e.response["Error"]["Code"] == "EntityNotFoundException":
                    return CheckpointResult(
                        status="fail",
                        error_reason=(
                            f"Glue table {DATABASE_NAME}.{tbl} does not exist. "
                            f"Re-run Task 1 to create both tables over the seeded Parquet."
                        ),
                    )
                return CheckpointResult(status="error", error_reason=str(e))

        # Iceberg table records its table_type. Defensive: confirm the
        # iceberg path has metadata objects, otherwise the Iceberg CREATE
        # silently picked the wrong location.
        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        meta_resp = s3_client.list_objects_v2(
            Bucket=BUCKET_NAME,
            Prefix=f"{ICEBERG_PREFIX}metadata/",
            MaxKeys=1,
        )
        if not meta_resp.get("Contents"):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"No metadata objects found at s3://{BUCKET_NAME}/"
                    f"{ICEBERG_PREFIX}metadata/. The Iceberg table either was "
                    f"not created with table_type='ICEBERG' or the LOCATION "
                    f"clause pointed somewhere else."
                ),
            )

        # Row counts match.
        hive_count_exec = run_athena(f"SELECT count(*) FROM {DATABASE_NAME}.{TABLE_HIVE}")
        hive_rows = fetch_rows(hive_count_exec["QueryExecutionId"])
        if len(hive_rows) < 2:
            return CheckpointResult(
                status="fail",
                error_reason=f"count(*) on {TABLE_HIVE} returned no rows.",
            )
        hive_count = int(hive_rows[1][0])

        ice_count_exec = run_athena(f"SELECT count(*) FROM {DATABASE_NAME}.{TABLE_ICEBERG}")
        ice_rows = fetch_rows(ice_count_exec["QueryExecutionId"])
        if len(ice_rows) < 2:
            return CheckpointResult(
                status="fail",
                error_reason=f"count(*) on {TABLE_ICEBERG} returned no rows.",
            )
        ice_count = int(ice_rows[1][0])

        if hive_count == 0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{TABLE_HIVE} has zero rows. Did the seed CTAS run? "
                    f"Check s3://{BUCKET_NAME}/{PARQUET_PREFIX} has Parquet files."
                ),
            )
        if hive_count != ice_count:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Row counts disagree: {TABLE_HIVE}={hive_count}, "
                    f"{TABLE_ICEBERG}={ice_count}. The INSERT INTO ... SELECT "
                    f"FROM {TABLE_HIVE} must complete before the checkpoint runs."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

Three pieces have to align before Checkpoint 1 can pass: the workgroup runs on Athena Engine version 3, both tables exist in the Glue catalog, and `events_iceberg` was populated from `events_hive` so the row counts match. The most common failure on first run is the workgroup defaulting to v2; Iceberg DDL silently fails on v2 with a syntax-looking error.

</details>

<details><summary>Hint 2 (direction)</summary>

The workgroup needs `EngineVersion={"SelectedEngineVersion": "Athena engine version 3"}` at creation, or an `update_work_group` call with `ConfigurationUpdates.EngineVersion` set the same way. The Hive table is a `CREATE EXTERNAL TABLE ... LOCATION 's3://.../parquet/'`. The Iceberg table is a `CREATE TABLE ... LOCATION 's3://.../iceberg/' TBLPROPERTIES ('table_type' = 'ICEBERG', 'format' = 'PARQUET')`. Once both tables exist, `INSERT INTO events_iceberg SELECT * FROM events_hive` populates the Iceberg side.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 1:

```python
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
glue = boto3.client(
    "glue",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

try:
    s3.create_bucket(Bucket=BUCKET_NAME)
    print(f"Bucket created: {BUCKET_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] in ("BucketAlreadyOwnedByYou", "BucketAlreadyExists"):
        print(f"Bucket {BUCKET_NAME} already owned; reusing.")
    else:
        raise

s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)

athena_results_uri = f"s3://{BUCKET_NAME}/{ATHENA_RESULTS_PREFIX}"
try:
    athena_client.create_work_group(
        Name=WORKGROUP_NAME,
        Configuration={
            "ResultConfiguration": {"OutputLocation": athena_results_uri},
            "EnforceWorkGroupConfiguration": True,
            "EngineVersion": {"SelectedEngineVersion": "Athena engine version 3"},
            "PublishCloudWatchMetricsEnabled": False,
        },
        Description=f"Lab workgroup for {LAB_ID}.",
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
    print(f"Athena workgroup created: {WORKGROUP_NAME} (Engine v3)")
except ClientError as e:
    if e.response["Error"]["Code"] == "InvalidRequestException":
        athena_client.update_work_group(
            WorkGroup=WORKGROUP_NAME,
            ConfigurationUpdates={
                "ResultConfigurationUpdates": {"OutputLocation": athena_results_uri},
                "EnforceWorkGroupConfiguration": True,
                "EngineVersion": {"SelectedEngineVersion": "Athena engine version 3"},
            },
            State="ENABLED",
        )
    else:
        raise

try:
    glue.create_database(
        DatabaseInput={"Name": DATABASE_NAME, "Description": f"Catalog for {LAB_ID}"},
    )
    glue.tag_resource(
        ResourceArn=f"arn:aws:glue:{REGION}:{ACCOUNT_ID}:database/{DATABASE_NAME}",
        TagsToAdd={LAB_TAG_KEY: LAB_TAG_VALUE},
    )
except ClientError as e:
    if e.response["Error"]["Code"] != "AlreadyExistsException":
        raise

# Seed Parquet via CTAS (the seed_ctas_sql string is defined above).
run_athena(seed_ctas_sql)
glue.delete_table(DatabaseName=DATABASE_NAME, Name=SEED_TABLE)

# events_hive (hive_ddl is defined above).
run_athena(hive_ddl)

# events_iceberg (iceberg_ddl and insert_sql are defined above).
run_athena(iceberg_ddl)
run_athena(insert_sql)
```

</details>

**Wallet check.** About 10 to 15 cents so far. The seed CTAS materialized ~1,000 Parquet rows; that single CTAS is the lab's only meaningful cost. The Hive and Iceberg DDL calls are free control-plane operations. Two `count(*)` queries scanned a fraction of a megabyte. Your coffee this morning still cost more.

## Task 2: Add a `loyalty_tier` column on both sides

First real schema evolution: the marketing team needs a `loyalty_tier` string on every event. Run `ALTER TABLE` on both formats and watch how each handles the column-added-after-the-fact case.

Build it in this order:

1. `ALTER TABLE events_hive ADD COLUMNS (loyalty_tier string)`. Hive's metastore accepts this immediately. The old Parquet files do not have the column on disk; Athena/Presto fills NULL for any row where the column is missing.
2. `ALTER TABLE events_iceberg ADD COLUMNS (loyalty_tier string)`. Iceberg's metadata layer records the new column and assigns it a stable column ID. Existing snapshots continue to read NULL for the new column; the change is invisible to ongoing queries.
3. Append one fresh row to each side, with `loyalty_tier` populated (use `'gold'`). On Hive, the append is `INSERT INTO events_hive (event_id, ..., loyalty_tier) VALUES (...)`. On Iceberg, the same shape.
4. Append a row to `outcome/results.json` recording the result. The helper takes care of the read-modify-write; you just record `step='add', hive='ok-no-rebuild', iceberg='ok-no-rebuild'`.

When this task ends, both tables have seven columns and old rows still query cleanly with NULL in the new column. The first evolution is the easy one. The next two are not.

In [ ]:
# NBVAL_SKIP
# Task 2: Add the loyalty_tier column on both sides, append a row with the
# new column populated, and write the first outcome row.

def read_outcomes() -> list[dict]:
    """Load outcome/results.json. Returns [] if the file does not exist yet."""
    try:
        resp = s3.get_object(Bucket=BUCKET_NAME, Key=OUTCOME_KEY)
        return json.loads(resp["Body"].read())
    except ClientError as e:
        if e.response["Error"]["Code"] == "NoSuchKey":
            return []
        raise


def write_outcome(step: str, hive_result: str, iceberg_result: str) -> None:
    """Append an outcome row to outcome/results.json with read-modify-write."""
    outcomes = read_outcomes()
    outcomes = [o for o in outcomes if o.get("step") != step]  # idempotent: replace if step exists
    outcomes.append({"step": step, "hive": hive_result, "iceberg": iceberg_result})
    s3.put_object(
        Bucket=BUCKET_NAME,
        Key=OUTCOME_KEY,
        Body=json.dumps(outcomes, indent=2).encode("utf-8"),
    )
    print(f"  outcome[{step}] = hive:{hive_result}, iceberg:{iceberg_result}")


# 1. Hive: ALTER TABLE ADD COLUMNS. Note the plural in Hive syntax under Athena.
hive_add_sql = (
    f"ALTER TABLE {DATABASE_NAME}.{TABLE_HIVE} "
    f"ADD COLUMNS (loyalty_tier string)"
)
# YOUR CODE: Run hive_add_sql using run_athena().
print(f"Hive ADD COLUMNS succeeded.")

# 2. Iceberg: ALTER TABLE ADD COLUMNS (plural; Iceberg shares the same
# COLUMNS keyword in Athena Engine v3).
iceberg_add_sql = (
    f"ALTER TABLE {DATABASE_NAME}.{TABLE_ICEBERG} "
    f"ADD COLUMNS (loyalty_tier string)"
)
# YOUR CODE: Run iceberg_add_sql using run_athena().
print(f"Iceberg ADD COLUMNS succeeded.")

# 3. Append a row with loyalty_tier populated to each side. The Hive append
# writes a new Parquet file that includes the loyalty_tier column on disk.
# Iceberg writes its own data file and updates the snapshot.
hive_insert_sql = f"""
INSERT INTO {DATABASE_NAME}.{TABLE_HIVE}
  (event_id, customer_id, event_ts, country, order_total, legacy_session_id, loyalty_tier)
VALUES (
  9001, 1042, from_iso8601_timestamp('2026-05-01T10:00:00Z'),
  'US', 250.0, 'sess_009001', 'gold'
)
"""
# YOUR CODE: Run hive_insert_sql using run_athena().

iceberg_insert_sql = f"""
INSERT INTO {DATABASE_NAME}.{TABLE_ICEBERG}
  (event_id, customer_id, event_ts, country, order_total, legacy_session_id, loyalty_tier)
VALUES (
  9001, 1042, from_iso8601_timestamp('2026-05-01T10:00:00Z'),
  'US', 250.0, 'sess_009001', 'gold'
)
"""
# YOUR CODE: Run iceberg_insert_sql using run_athena().
print(f"New row appended to both tables with loyalty_tier='gold'.")

# 4. Record the outcome.
write_outcome("add", "ok-no-rebuild", "ok-no-rebuild")

print()
print("Both tables now have a loyalty_tier column. Old rows read NULL; new row reads 'gold'.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: loyalty_tier column exists in both Glue table schemas; the
# appended row reads back 'gold' on both sides; at least one old row reads
# NULL for loyalty_tier on both sides; outcome/results.json carries the
# add row. No comparison logic; validator inspects each side independently.

def checkpoint_2(session):
    try:
        glue_client = boto3.client(
            "glue",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        # Hive side: loyalty_tier in column list.
        hive_meta = glue_client.get_table(DatabaseName=DATABASE_NAME, Name=TABLE_HIVE)
        hive_cols = {c["Name"] for c in hive_meta["Table"]["StorageDescriptor"]["Columns"]}
        if "loyalty_tier" not in hive_cols:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"loyalty_tier not present in {TABLE_HIVE} StorageDescriptor. "
                    f"Run ALTER TABLE {TABLE_HIVE} ADD COLUMNS (loyalty_tier string)."
                ),
            )

        # Iceberg side: loyalty_tier in column list.
        ice_meta = glue_client.get_table(DatabaseName=DATABASE_NAME, Name=TABLE_ICEBERG)
        ice_cols = {c["Name"] for c in ice_meta["Table"]["StorageDescriptor"]["Columns"]}
        if "loyalty_tier" not in ice_cols:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"loyalty_tier not present in {TABLE_ICEBERG} StorageDescriptor. "
                    f"Run ALTER TABLE {TABLE_ICEBERG} ADD COLUMNS (loyalty_tier string)."
                ),
            )

        # Appended row reads back 'gold' on both sides.
        for tbl in (TABLE_HIVE, TABLE_ICEBERG):
            sql = (
                f"SELECT loyalty_tier FROM {DATABASE_NAME}.{tbl} "
                f"WHERE event_id = 9001"
            )
            exec_ = run_athena(sql)
            rows = fetch_rows(exec_["QueryExecutionId"])
            if len(rows) < 2:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"No row with event_id=9001 found in {tbl}. The new "
                        f"row with loyalty_tier='gold' must be inserted."
                    ),
                )
            value = rows[1][0]
            if value != "gold":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"event_id=9001 in {tbl} has loyalty_tier={value!r}, "
                        f"expected 'gold'."
                    ),
                )

        # Old rows read NULL on both sides for loyalty_tier. Take any row that
        # is not the appended one.
        for tbl in (TABLE_HIVE, TABLE_ICEBERG):
            sql = (
                f"SELECT loyalty_tier FROM {DATABASE_NAME}.{tbl} "
                f"WHERE event_id = 1 LIMIT 1"
            )
            exec_ = run_athena(sql)
            rows = fetch_rows(exec_["QueryExecutionId"])
            if len(rows) < 2:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"No row with event_id=1 in {tbl}. The seed CTAS in "
                        f"Task 1 should have written event_id values 1..1000."
                    ),
                )
            # Athena renders SQL NULL as the empty string in VarCharValue. The
            # cell may also be missing entirely; both are NULL semantics.
            value = rows[1][0] if rows[1] else ""
            if value != "":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Old row event_id=1 in {tbl} returned loyalty_tier={value!r}, "
                        f"expected NULL because the column was added after the row."
                    ),
                )

        # outcome/results.json has the add row.
        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        try:
            obj = s3_client.get_object(Bucket=BUCKET_NAME, Key=OUTCOME_KEY)
            outcomes = json.loads(obj["Body"].read())
        except ClientError as e:
            if e.response["Error"]["Code"] == "NoSuchKey":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"outcome/results.json does not exist at "
                        f"s3://{BUCKET_NAME}/{OUTCOME_KEY}. Call write_outcome('add', ...)"
                        f" at the end of Task 2."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))
        add_row = next((o for o in outcomes if o.get("step") == "add"), None)
        if add_row is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"outcome/results.json is missing the step='add' row."
                ),
            )
        if add_row.get("hive") != "ok-no-rebuild" or add_row.get("iceberg") != "ok-no-rebuild":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"outcome[add] should record hive='ok-no-rebuild' and "
                    f"iceberg='ok-no-rebuild'. Found: {add_row}."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

ADD COLUMN behaves nearly identically on both formats. The checkpoint fails when either the ALTER did not run, the new row was not appended with `loyalty_tier='gold'` and `event_id=9001`, or the outcome JSON was not written. Check that both ALTERs ran in Athena (not just at the catalog level via Glue) and that the INSERT INTOs followed.

</details>

<details><summary>Hint 2 (direction)</summary>

Athena DDL for both formats uses `ALTER TABLE <db>.<table> ADD COLUMNS (loyalty_tier string)`, note the plural `COLUMNS`. Iceberg under Athena Engine v3 accepts the same keyword. The INSERT INTO that appends the new row needs the full column list in the parenthesized projection, including `loyalty_tier`, and a `VALUES (...)` row with `'gold'` in the last position. The outcome write is one `write_outcome('add', 'ok-no-rebuild', 'ok-no-rebuild')` call.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 2:

```python
hive_add_sql = (
    f"ALTER TABLE {DATABASE_NAME}.{TABLE_HIVE} "
    f"ADD COLUMNS (loyalty_tier string)"
)
run_athena(hive_add_sql)

iceberg_add_sql = (
    f"ALTER TABLE {DATABASE_NAME}.{TABLE_ICEBERG} "
    f"ADD COLUMNS (loyalty_tier string)"
)
run_athena(iceberg_add_sql)

hive_insert_sql = f"""
INSERT INTO {DATABASE_NAME}.{TABLE_HIVE}
  (event_id, customer_id, event_ts, country, order_total, legacy_session_id, loyalty_tier)
VALUES (
  9001, 1042, from_iso8601_timestamp('2026-05-01T10:00:00Z'),
  'US', 250.0, 'sess_009001', 'gold'
)
"""
run_athena(hive_insert_sql)

iceberg_insert_sql = f"""
INSERT INTO {DATABASE_NAME}.{TABLE_ICEBERG}
  (event_id, customer_id, event_ts, country, order_total, legacy_session_id, loyalty_tier)
VALUES (
  9001, 1042, from_iso8601_timestamp('2026-05-01T10:00:00Z'),
  'US', 250.0, 'sess_009001', 'gold'
)
"""
run_athena(iceberg_insert_sql)

write_outcome("add", "ok-no-rebuild", "ok-no-rebuild")
```

</details>

**Wallet check.** Still under 20 cents. Two ALTERs, two single-row INSERTs, and four SELECTs that scan kilobytes. Iceberg also wrote a new metadata snapshot to the `iceberg/metadata/` prefix, which is a few KB of JSON. The meter has barely budged.

## Task 3: Rename `order_total` to `order_amount_usd`

Second evolution: finance asked the team to use a clearer column name. On Iceberg, `ALTER TABLE ... RENAME COLUMN` is a metadata-only change and the read keeps working because Iceberg's metadata layer maps stable column IDs to current column names. On Hive, the same rename `succeeds` at the metastore but the read breaks because Parquet files store columns by name, not by position, and the on-disk column name does not match the new metastore name.

This is the teaching moment of the lab. The Hive metastore reports success. Then `SELECT order_amount_usd FROM events_hive` returns all NULLs. The metastore and the data files disagree about who knows the truth.

Build it in this order:

1. Iceberg: `ALTER TABLE events_iceberg RENAME COLUMN order_total TO order_amount_usd`. This is the v3-only syntax Athena supports for Iceberg.
2. Hive: `ALTER TABLE events_hive CHANGE COLUMN order_total order_amount_usd double`. This is Hive's rename equivalent. Athena accepts it. The metastore now reports `order_amount_usd` as the column name.
3. Run `SELECT order_amount_usd FROM events_<table> LIMIT 5` against each side. On Iceberg, real numbers come back. On Hive, NULLs come back because the Parquet file's column is still named `order_total` on disk and the reader cannot find `order_amount_usd`.
4. Record `outcome` step `'rename'` with `hive='required-rebuild', iceberg='ok-no-rebuild'`. The Hive failure is not a bug; it is the structural difference between metastore-only renames and metadata-tracked renames.

In [ ]:
# NBVAL_SKIP
# Task 3: Rename order_total to order_amount_usd on both sides. Iceberg's
# rename is metadata-only and the read keeps working. Hive's rename succeeds
# at the metastore but the Parquet column is still named order_total on disk,
# so SELECT order_amount_usd FROM events_hive returns NULLs.

# 1. Iceberg rename: metadata-only, reads keep working.
iceberg_rename_sql = (
    f"ALTER TABLE {DATABASE_NAME}.{TABLE_ICEBERG} "
    f"RENAME COLUMN order_total TO order_amount_usd"
)
# YOUR CODE: Run iceberg_rename_sql using run_athena().
print(f"Iceberg rename succeeded (metadata-only).")

# 2. Hive rename: metastore-only. CHANGE COLUMN is Hive's rename syntax in
# Athena. The data type must be specified again even when unchanged.
hive_rename_sql = (
    f"ALTER TABLE {DATABASE_NAME}.{TABLE_HIVE} "
    f"CHANGE COLUMN order_total order_amount_usd double"
)
# YOUR CODE: Run hive_rename_sql using run_athena().
print(f"Hive rename succeeded at the metastore. Reads against the new name will surprise you.")

# 3. Verify each side independently with a SELECT.
iceberg_check_sql = (
    f"SELECT order_amount_usd FROM {DATABASE_NAME}.{TABLE_ICEBERG} "
    f"WHERE event_id <= 5 ORDER BY event_id"
)
ice_exec = run_athena(iceberg_check_sql)
ice_rows = fetch_rows(ice_exec["QueryExecutionId"])
print("Iceberg SELECT order_amount_usd LIMIT 5:")
for r in ice_rows[1:]:
    print("  ", r)

hive_check_sql = (
    f"SELECT order_amount_usd FROM {DATABASE_NAME}.{TABLE_HIVE} "
    f"WHERE event_id <= 5 ORDER BY event_id"
)
hive_exec = run_athena(hive_check_sql)
hive_rows = fetch_rows(hive_exec["QueryExecutionId"])
print("Hive SELECT order_amount_usd LIMIT 5 (expect NULLs because Parquet stores by name):")
for r in hive_rows[1:]:
    print("  ", r)

# 4. Record the outcome.
# YOUR CODE: Call write_outcome with step='rename', hive='required-rebuild',
# iceberg='ok-no-rebuild'.

print()
print("Rename divergence captured. The Hive metastore reports success; the file system disagrees.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: Iceberg's rename actually works (column renamed AND values
# read back correctly). Hive's rename is metastore-only; the SELECT on the
# new column name returns NULLs. outcome/results.json records the divergence.
# No comparison logic in validator; each side checked independently.

def checkpoint_3(session):
    try:
        glue_client = boto3.client(
            "glue",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        # Both catalogs reflect the rename in their column list (Hive's
        # metastore-only rename still updates the Glue table definition).
        for tbl in (TABLE_HIVE, TABLE_ICEBERG):
            meta = glue_client.get_table(DatabaseName=DATABASE_NAME, Name=tbl)
            cols = {c["Name"] for c in meta["Table"]["StorageDescriptor"]["Columns"]}
            if "order_amount_usd" not in cols:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"{tbl} does not have order_amount_usd in the Glue "
                        f"StorageDescriptor. Run the rename ALTER on this side."
                    ),
                )
            if "order_total" in cols:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"{tbl} still has order_total in the Glue StorageDescriptor. "
                        f"Rename did not run; expected order_total to be replaced "
                        f"by order_amount_usd."
                    ),
                )

        # Iceberg side: SELECT order_amount_usd returns real values (non-NULL
        # for the seed rows, which had order_total set).
        ice_exec = run_athena(
            f"SELECT order_amount_usd FROM {DATABASE_NAME}.{TABLE_ICEBERG} "
            f"WHERE event_id <= 5 AND order_amount_usd IS NOT NULL"
        )
        ice_rows = fetch_rows(ice_exec["QueryExecutionId"])
        # rows[0] is header; rows[1:] are data.
        if len(ice_rows) < 2:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Iceberg SELECT order_amount_usd returned no non-NULL rows. "
                    f"The rename should be metadata-only and existing values "
                    f"should read through."
                ),
            )

        # Hive side: SELECT order_amount_usd returns all NULLs for the seed
        # rows, because the Parquet column on disk is still named order_total.
        hive_exec = run_athena(
            f"SELECT order_amount_usd FROM {DATABASE_NAME}.{TABLE_HIVE} "
            f"WHERE event_id <= 5 AND order_amount_usd IS NOT NULL"
        )
        hive_rows = fetch_rows(hive_exec["QueryExecutionId"])
        # If Hive returns rows, the rename did something unexpected (the data
        # files were rewritten, which is not the default Athena behavior). The
        # divergence is the lab's teaching point, so any non-NULL means the
        # student's setup deviates from the expected Hive semantics.
        if len(hive_rows) >= 2:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Hive SELECT order_amount_usd returned non-NULL values "
                    f"for event_id <= 5. Expected NULLs because Parquet stores "
                    f"by name and the on-disk column is still order_total. "
                    f"Did the Parquet files at s3://{BUCKET_NAME}/{PARQUET_PREFIX} "
                    f"get rewritten by an external process?"
                ),
            )

        # outcome/results.json has the rename row.
        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        try:
            obj = s3_client.get_object(Bucket=BUCKET_NAME, Key=OUTCOME_KEY)
            outcomes = json.loads(obj["Body"].read())
        except ClientError as e:
            return CheckpointResult(status="error", error_reason=str(e))
        rename_row = next((o for o in outcomes if o.get("step") == "rename"), None)
        if rename_row is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"outcome/results.json is missing the step='rename' row. "
                    f"Call write_outcome('rename', 'required-rebuild', 'ok-no-rebuild')."
                ),
            )
        if rename_row.get("hive") != "required-rebuild" or rename_row.get("iceberg") != "ok-no-rebuild":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"outcome[rename] should record hive='required-rebuild' and "
                    f"iceberg='ok-no-rebuild'. Found: {rename_row}."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

Hive renames at the metastore but Parquet column names are baked into the file footers. The checkpoint fails when either the rename ALTER did not run on one side, the outcome JSON was not updated, or the Hive read against the new column name unexpectedly returned non-NULL values (which would mean something rewrote the Parquet files behind the lab's back).

</details>

<details><summary>Hint 2 (direction)</summary>

Athena reads Hive columns by name from Parquet, so a metastore-only rename produces silent NULLs unless you rewrite the file with `INSERT OVERWRITE` or a CTAS. The lab's teaching point is that the rename does not auto-rewrite. Use `ALTER TABLE events_iceberg RENAME COLUMN order_total TO order_amount_usd` on the Iceberg side and `ALTER TABLE events_hive CHANGE COLUMN order_total order_amount_usd double` on the Hive side (Hive's CHANGE COLUMN syntax requires re-stating the type). Then record `'required-rebuild'` for Hive and `'ok-no-rebuild'` for Iceberg in the outcome JSON.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 3:

```python
iceberg_rename_sql = (
    f"ALTER TABLE {DATABASE_NAME}.{TABLE_ICEBERG} "
    f"RENAME COLUMN order_total TO order_amount_usd"
)
run_athena(iceberg_rename_sql)

hive_rename_sql = (
    f"ALTER TABLE {DATABASE_NAME}.{TABLE_HIVE} "
    f"CHANGE COLUMN order_total order_amount_usd double"
)
run_athena(hive_rename_sql)

ice_exec = run_athena(
    f"SELECT order_amount_usd FROM {DATABASE_NAME}.{TABLE_ICEBERG} "
    f"WHERE event_id <= 5 ORDER BY event_id"
)
print("Iceberg returns real values.")

hive_exec = run_athena(
    f"SELECT order_amount_usd FROM {DATABASE_NAME}.{TABLE_HIVE} "
    f"WHERE event_id <= 5 ORDER BY event_id"
)
print("Hive returns NULLs because Parquet stores by name.")

write_outcome("rename", "required-rebuild", "ok-no-rebuild")
```

The cost of "fixing" the Hive side is rewriting the entire Parquet dataset: at 500M rows and roughly $0.10 per GB scanned in the rewrite path, an Iceberg-equivalent rename on Hive costs about $50 per evolution. Multiply by every six weeks. Multiply by every table.

</details>

**Wallet check.** Still under 25 cents. Two more ALTERs (free), two SELECTs that scan a few rows each, and one S3 PutObject for the outcome JSON. The expensive part of fixing Hive's rename would be the rewrite, which this lab is deliberately not doing. The whole point is to show what the rewrite would cost.

## Task 4: Drop `legacy_session_id`, reorder columns, then aggregate the comparison

Two more evolutions and the lab is done.

1. **Drop column.** Both formats let you drop a column. Hive's drop is metastore-only (the column still exists in the Parquet files on disk; the reader just stops asking for it). Iceberg's drop is metadata-only and marks the column ID retired. Both pass without rebuilding. Record `step='drop', hive='ok-no-rebuild', iceberg='ok-no-rebuild'`.
2. **Reorder columns.** Iceberg supports `ALTER TABLE ... CHANGE COLUMN ... AFTER ...` and the reorder is metadata-only because Iceberg tracks columns by stable IDs, not by position. Hive's reorder is meaningless: the Parquet file stores columns by name, and the metastore's column order does not change the on-disk layout. Athena reads Hive columns by name regardless of the metastore order. Record `step='reorder', hive='required-rebuild', iceberg='ok-no-rebuild'`.
3. **Aggregate.** With four outcome rows now in `outcome/results.json`, compute the comparison metric: `"Hive: 2 of 4 evolutions succeeded without rebuild. Iceberg: 4 of 4 succeeded without rebuild."` Print it. The checkpoint reads the metric off `outcome/results.json`.

After this task, you have a defensible artifact: real outcomes, real cost implications, ready for the standup tomorrow.

In [ ]:
# NBVAL_SKIP
# Task 4: Drop legacy_session_id, reorder columns on both sides, then
# aggregate the four outcome rows and print the comparison metric.

# 1. Drop legacy_session_id on both sides.
hive_drop_sql = (
    f"ALTER TABLE {DATABASE_NAME}.{TABLE_HIVE} "
    f"DROP COLUMN legacy_session_id"
)
# YOUR CODE: Run hive_drop_sql using run_athena().

iceberg_drop_sql = (
    f"ALTER TABLE {DATABASE_NAME}.{TABLE_ICEBERG} "
    f"DROP COLUMN legacy_session_id"
)
# YOUR CODE: Run iceberg_drop_sql using run_athena().
print(f"Drop column ran on both sides.")

# YOUR CODE: Call write_outcome with step='drop', hive='ok-no-rebuild',
# iceberg='ok-no-rebuild'.

# 2. Reorder: move country to be right after event_id on Iceberg via
# CHANGE COLUMN ... AFTER. Iceberg's reorder is metadata-only.
iceberg_reorder_sql = (
    f"ALTER TABLE {DATABASE_NAME}.{TABLE_ICEBERG} "
    f"CHANGE COLUMN country country string AFTER event_id"
)
# YOUR CODE: Run iceberg_reorder_sql using run_athena().

# Hive: the equivalent reorder syntax exists in Athena's Hive DDL but it only
# updates the metastore column order. The Parquet files still encode columns
# by name. Reordering at the metastore does not change query results because
# Athena resolves columns by name from Parquet regardless. This is the second
# evolution that "succeeds" at the metastore but does not actually rearrange
# the data layout the way Iceberg does.
hive_reorder_sql = (
    f"ALTER TABLE {DATABASE_NAME}.{TABLE_HIVE} "
    f"CHANGE COLUMN country country string AFTER event_id"
)
# YOUR CODE: Run hive_reorder_sql using run_athena().
print(f"Reorder ran on both sides. Iceberg reordered for real; Hive only at the metastore.")

# YOUR CODE: Call write_outcome with step='reorder', hive='required-rebuild',
# iceberg='ok-no-rebuild'.

# 3. Aggregate and print the comparison metric.
final_outcomes = read_outcomes()
hive_no_rebuild = sum(1 for o in final_outcomes if o["hive"] == "ok-no-rebuild")
ice_no_rebuild = sum(1 for o in final_outcomes if o["iceberg"] == "ok-no-rebuild")
total = len(final_outcomes)

metric = (
    f"Hive: {hive_no_rebuild} of {total} evolutions succeeded without rebuild. "
    f"Iceberg: {ice_no_rebuild} of {total} succeeded without rebuild. "
    f"The two Hive failures are rename and reorder; both require either a "
    f"stale metastore or a Parquet rewrite."
)
print()
print("Comparison metric:")
print("  " + metric)
print()
print("outcome/results.json:")
for row in final_outcomes:
    print(f"  {row}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: outcome/results.json has all four step rows with the expected
# hive/iceberg outcomes. The comparison metric is a display artifact; the
# validator asserts the underlying counts come out as 2/4 vs 4/4 but does
# NOT itself compute which format "wins" (per Compare-it pattern blueprint
# Section 21: validators are infrastructure-objective, never comparative).

def checkpoint_4(session):
    try:
        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        try:
            obj = s3_client.get_object(Bucket=BUCKET_NAME, Key=OUTCOME_KEY)
            outcomes = json.loads(obj["Body"].read())
        except ClientError as e:
            if e.response["Error"]["Code"] == "NoSuchKey":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"outcome/results.json does not exist at "
                        f"s3://{BUCKET_NAME}/{OUTCOME_KEY}."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        expected = {
            "add": ("ok-no-rebuild", "ok-no-rebuild"),
            "rename": ("required-rebuild", "ok-no-rebuild"),
            "drop": ("ok-no-rebuild", "ok-no-rebuild"),
            "reorder": ("required-rebuild", "ok-no-rebuild"),
        }
        steps_seen = {o.get("step"): o for o in outcomes}

        missing = [s for s in expected if s not in steps_seen]
        if missing:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"outcome/results.json is missing rows for: {missing}. "
                    f"Each task must call write_outcome(step, hive_result, iceberg_result)."
                ),
            )

        for step, (hive_expected, ice_expected) in expected.items():
            row = steps_seen[step]
            if row.get("hive") != hive_expected:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"outcome[{step}].hive is {row.get('hive')!r}, "
                        f"expected {hive_expected!r}."
                    ),
                )
            if row.get("iceberg") != ice_expected:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"outcome[{step}].iceberg is {row.get('iceberg')!r}, "
                        f"expected {ice_expected!r}."
                    ),
                )

        # Independently verify the drop reached both catalogs.
        glue_client = boto3.client(
            "glue",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        for tbl in (TABLE_HIVE, TABLE_ICEBERG):
            meta = glue_client.get_table(DatabaseName=DATABASE_NAME, Name=tbl)
            cols = {c["Name"] for c in meta["Table"]["StorageDescriptor"]["Columns"]}
            if "legacy_session_id" in cols:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"{tbl} still has legacy_session_id in its Glue StorageDescriptor. "
                        f"The DROP COLUMN must run on this side."
                    ),
                )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

Drop is metastore-only on Hive; the Parquet still has the column on disk but the reader stops asking for it, so the practical effect is identical to Iceberg. Reorder on Hive is meaningless because Parquet stores columns by name and Athena resolves by name regardless of metastore order. Reorder on Iceberg is metadata-only and works for real. The checkpoint fails when either the DROP did not run, the reorder ALTER did not run, or the outcome JSON does not contain all four step rows.

</details>

<details><summary>Hint 2 (direction)</summary>

Use `ALTER TABLE ... DROP COLUMN legacy_session_id` on both sides. For the reorder, use `ALTER TABLE ... CHANGE COLUMN country country string AFTER event_id` on both sides; Iceberg actually rearranges the columns by ID, Hive only updates the metastore order. Record `drop` as `ok-no-rebuild` on both sides; record `reorder` as `required-rebuild` for Hive and `ok-no-rebuild` for Iceberg. After all four outcomes are recorded, the comparison metric reads 2-of-4 vs 4-of-4.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 4:

```python
hive_drop_sql = (
    f"ALTER TABLE {DATABASE_NAME}.{TABLE_HIVE} "
    f"DROP COLUMN legacy_session_id"
)
run_athena(hive_drop_sql)

iceberg_drop_sql = (
    f"ALTER TABLE {DATABASE_NAME}.{TABLE_ICEBERG} "
    f"DROP COLUMN legacy_session_id"
)
run_athena(iceberg_drop_sql)

write_outcome("drop", "ok-no-rebuild", "ok-no-rebuild")

iceberg_reorder_sql = (
    f"ALTER TABLE {DATABASE_NAME}.{TABLE_ICEBERG} "
    f"CHANGE COLUMN country country string AFTER event_id"
)
run_athena(iceberg_reorder_sql)

hive_reorder_sql = (
    f"ALTER TABLE {DATABASE_NAME}.{TABLE_HIVE} "
    f"CHANGE COLUMN country country string AFTER event_id"
)
run_athena(hive_reorder_sql)

write_outcome("reorder", "required-rebuild", "ok-no-rebuild")

final_outcomes = read_outcomes()
hive_no_rebuild = sum(1 for o in final_outcomes if o["hive"] == "ok-no-rebuild")
ice_no_rebuild = sum(1 for o in final_outcomes if o["iceberg"] == "ok-no-rebuild")
total = len(final_outcomes)
metric = (
    f"Hive: {hive_no_rebuild} of {total} evolutions succeeded without rebuild. "
    f"Iceberg: {ice_no_rebuild} of {total} succeeded without rebuild. "
    f"The two Hive failures are rename and reorder; both require either a "
    f"stale metastore or a Parquet rewrite."
)
print(metric)
```

</details>

**Wallet check.** Total session damage lands around 30 to 45 cents. Four ALTERs (free), zero new data on disk, two outcome JSON writes (rounding error). The expensive operation, in production, would be the Hive rewrite to make rename and reorder real; this lab deliberately did not run it so you can see the gap on paper. Your morning coffee cost more.

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in reverse-creation
# order. No critical (hourly-billed) resources in this lab. Canonical
# summary per RESOURCE-SAFETY-SPEC Rule 6. sys.exit(1) on dirty cleanup.

import sys

result = run_cleanup(CLEANUP_MANIFEST)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids: set[str] = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: approximately $0.20 to $0.45.** The seed CTAS was the only material cost. Athena scans across all four evolutions added up to under 5 MB. The bucket and Glue catalog entries were destroyed by the cleanup cell above; nothing keeps accruing. Check your AWS Billing console in 24 hours to confirm the exact number. Your coffee this morning still cost more than your AWS bill from this session.

## These are not graded. They are for you.

Three questions worth sitting with before the next standup.

1. The Hive rename `succeeded` at the metastore but the read broke. In your own words, what is the metastore actually tracking, and what is the Parquet file actually storing? Why does Iceberg's metadata layer make the difference, and what would the equivalent fix on Hive cost on a 500-million-row table? Cite the rewrite-cost math.

2. Walk through the four schema-evolution outcomes you captured. Which two evolutions are "safe" on both formats, and what is the on-disk reason they are safe? Which two evolutions diverge, and what is the structural reason for the divergence? If your team committed to Iceberg today, what is the smallest set of patterns you would still need to teach because they are not free even on Iceberg?

3. The product manager has told you the schema is changing every six weeks for the next two quarters. The dataset is 500M rows growing at 50M per month. Argue both sides: when would you still pick Hive over Iceberg, and when does Iceberg's metadata-only evolution clearly justify the migration cost? Make the cost case in dollars and engineer-hours, not in feelings about the platform team.

## Exam-Style Practice

**Q1.** Your product team is rolling out a new schema every six weeks. The dataset has 500 million rows in Parquet, growing at 50 million rows per month. The two most common evolutions over the next two quarters will be column renames and column reorders. Which table format earns the migration cost given these constraints?

A) Hive-style external tables, because rewrites of 500M-row Parquet datasets are tolerable at this volume.

B) Iceberg, because column rename and column reorder are metadata-only operations and the dataset never needs a rewrite for a typed schema change.

C) Parquet without a catalog, because schema-on-read is more flexible and lets each consumer interpret columns independently.

D) JSON files over Iceberg, because the document format is more tolerant of schema drift than columnar formats.

<details><summary>Show answer</summary>

**B**.

- A) Wrong because a 500-million-row rewrite costs roughly $50 per evolution at Athena's $5/TB scan rate (assuming a 10 GB Parquet footprint after compression) and pinning that to a six-week cadence makes the migration cost the cheaper option within two evolutions; rewrites at this volume are NOT "tolerable" when there is a metadata-only path available.
- B) Right because this is the constraint scenario the lab simulated. Iceberg's metadata layer tracks columns by stable IDs; rename and reorder change only the metadata snapshot and the existing Parquet data files are reused unchanged. Both evolutions cost milliseconds in Iceberg and cost a full rewrite in Hive.
- C) Wrong because schema-on-read pushes the cost onto every consumer (each query has to know the file shape) and explicitly forbids the centralized catalog enforcement that Iceberg provides; it scales worst with team count.
- D) Wrong because JSON over Iceberg defeats the columnar storage benefits of Parquet (compression, projection pushdown, predicate pushdown) and Iceberg's value-add is the same regardless of the underlying file format; the columnar+metadata combination is what makes Iceberg cheap.

</details>

**Q2.** An engineer runs `ALTER TABLE events_hive CHANGE COLUMN order_total order_amount_usd double` against a Hive-style external table over Parquet. The metastore reports success. The next morning, a downstream Tableau dashboard sees the same query return all NULLs where it used to return totals. What happened?

A) The Parquet files were corrupted during the ALTER and need to be restored from backup.

B) The Hive metastore rename is name-only; Athena reads Parquet columns by name and the on-disk column is still `order_total`, so `SELECT order_amount_usd` finds no matching column and returns NULL.

C) Athena workgroups need to be on Engine v2 for column renames; the workgroup got bumped to v3 overnight and v3 silently nulls renamed columns.

D) The Glue Data Catalog needs to be re-indexed via a crawler before any renamed column reads correctly.

<details><summary>Show answer</summary>

**B**.

- A) Wrong because nothing in the ALTER pathway rewrites or touches Parquet data files; the metastore is a separate control-plane store. A backup restore would not change the behavior.
- B) Right because this is the rename divergence the lab demonstrated. Hive renames at the metastore but the Parquet column footer still encodes the old name; Athena (and any Parquet reader) maps the query's column name to the on-disk column name. The fix is either a full rewrite (`INSERT OVERWRITE`) of the Parquet files or migration to Iceberg's metadata layer.
- C) Wrong because Engine v3 has nothing to do with rename semantics on Hive tables. The Parquet column-by-name resolution is a property of the file format, not the engine version.
- D) Wrong because Glue crawlers crawl files to infer schema; they do not change how a query resolves a column name against a Parquet file footer. Re-running a crawler would re-discover `order_total` in the file and overwrite the metastore back to its original column name.

</details>

**Q3.** Your team is building a new analytics dataset on S3. The schema is stable, the data is small (under 50 GB total), and the table will only ever be appended to. No column renames, no reorders, no drops expected for the next two years. Which format choice is the most defensible?

A) Iceberg, because Iceberg is always the right call regardless of access pattern.

B) Hive-style external Parquet, because the metadata overhead of Iceberg's manifest files outweighs the benefit when no evolution is expected.

C) CSV without a catalog, because the dataset is small enough that file format does not matter.

D) Delta Lake, because Delta is interchangeable with Iceberg and the team can pick either.

<details><summary>Show answer</summary>

**B**.

- A) Wrong because Iceberg's benefits (metadata-only schema evolution, hidden partitioning, time travel) all require maintaining manifest files and snapshot metadata; on a stable append-only schema at 50 GB, those benefits do not pay for the engineering overhead.
- B) Right because this is the inverse of the lab's constraint scenario. When schema is stable and access is append-only, Hive-style external Parquet is simpler, has zero metadata overhead, and works everywhere. The lab's takeaway is constraint-driven format choice, not "Iceberg always."
- C) Wrong because CSV has no compression, no columnar projection pushdown, and no predicate pushdown; even at 50 GB this triples the scan cost of every query.
- D) Wrong because Delta and Iceberg are NOT interchangeable; the file formats, manifest layouts, and engine support differ, and Athena's Iceberg support does not extend to Delta. Picking one is a real decision with platform consequences.

</details>